# ADC帧解析与新样本质量检查
默认合成数据，不采实际ADC、不控制激励电流或电机。相同码值不代表重复样本；相同序号才被拒绝。真实source必须提供原子读取的序号+帧，不能用PS循环计数伪造新样本。


In [ ]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
if not (ROOT/'scripts/adc_capture.py').is_file(): ROOT = ROOT.parent
assert (ROOT/'scripts/adc_capture.py').is_file()
sys.path.insert(0, str(ROOT/'scripts'))
from adc_capture import AdcConfig, decode_frame, analyze_records, capture, CaptureError


## 1. 可调位域与前端换算
转换使用2^N分母：最大码距上限一个LSB。不硬编码2.5V偏置或四探针校准。input范围描述ADC传输映射，gain描述前端增益，signal=(adc-bias)/gain。具体芯片可能用不同传输规则，必须先核对手册。


In [ ]:
FRAME_BITS = 16
CODE_BITS = 12
CODE_LSB = 0
CODING = 'straight'  # straight / twos / offset
INPUT_MIN_V = 0.0
INPUT_MAX_V = 4.096
FRONTEND_BIAS_V = 1.0
FRONTEND_GAIN = 2.0
SEQUENCE_BITS = 32
DISCARD_FIRST = 2
REQUIRED_SAMPLES = 8
MAX_STD_V = None  # 不限制噪声时None；需用实测数据选门限
config = AdcConfig(frame_bits=FRAME_BITS, code_bits=CODE_BITS, code_lsb=CODE_LSB, coding=CODING, input_min_v=INPUT_MIN_V, input_max_v=INPUT_MAX_V, bias_v=FRONTEND_BIAS_V, gain=FRONTEND_GAIN, sequence_bits=SEQUENCE_BITS, max_std_v=MAX_STD_V)
print('sample code:', decode_frame(2000, config))


## 2. 分析已有records
输入JSON的每条记录有sequence、frame，可选status_ok。缺序号不会被宣称为新样本；通道、饱和、缺失、乱序及超时独立判断。


In [ ]:
RECORDS_JSON = None
if RECORDS_JSON is None:
    records = [dict(sequence=i, frame=2000) for i in range(12)]
    source = 'SYNTHETIC'
else:
    records = json.loads(Path(RECORDS_JSON).read_text(encoding='utf-8'))
    source = 'USER_RECORDS_NOT_AUTHENTICATED'
result = analyze_records(records, config, discard_first=DISCARD_FIRST, required_samples=REQUIRED_SAMPLES)
result['source'] = source
print({k:v for k,v in result.items() if k!='samples'})


## 3. Provider采集接口离线演示
此处provider只是列表；真实provider需自行实现有界/非阻塞读取和原子快照，工具不会猜MMIO地址。失败时CaptureError.partial保留已有样本与质量标志。


In [ ]:
iterator = iter(records)
try:
    captured = capture(lambda: next(iterator), config, required_samples=REQUIRED_SAMPLES, discard_first=DISCARD_FIRST, timeout_s=1.0, poll_s=0.001)
    print('Offline capture valid:', captured['valid_count'])
except CaptureError as exc:
    print('FAILED:', str(exc), exc.partial['quality_flags'])
    raise


In [ ]:
SAVE_REPORT = False
if SAVE_REPORT:
    (ROOT/'examples/adc_debug_report.json').write_text(json.dumps(result, indent=2, allow_nan=False)+'\n', encoding='utf-8')
print('No ADC, current or motion hardware accessed.')
